# Download EMIMesh Data

This notebook downloads the EMIMesh repository and creates a dataset of meshed single cells and multi-cell cubes.

## Clone the modified EMIMesh repository
The ! runs a command the same way as writing it in the terminal

In [ ]:
import os

# Clone or update the modified emimesh repository
REPO_URL = "https://github.com/SamiLaubo/emimesh.git"
DEST_DIR = "../emimesh"

if not os.path.exists(DEST_DIR):
    print(f"Cloning {REPO_URL} into {DEST_DIR}...")
    !git clone $REPO_URL $DEST_DIR
else:
    print(f"Updating {DEST_DIR}...")
    !git -C $DEST_DIR pull

## Create environment

Install [conda](https://www.anaconda.com/docs/getting-started/installation) as your python environment manager. Same setup as EMIMesh.

In [ ]:
# Install snakemake (using conda)
!conda create -c conda-forge -c bioconda -n snakemake snakemake snakemake-storage-plugin-http snakemake-executor-plugin-cluster-generic -y

Activate this environment in this notebook and use it when running the following codes.

## Create configuration files for different data

The dataset consists of the following configurations.

| **Name** | **# Cells** | **Resolution** | **# Samples** | **Cell Index** |
| --- | --- | --- | --- | --- |
| One neuron | 1 | All (1,2,3,4,5) | 10 | Sampled |
| One astrocyte | 1 | All (1,2,3,4,5) | 10 | Sampled |
| Microglia | 1 | All (1,2,3,4,5) | 5 | Sampled |
| Oligo | 1 | All (1,2,3,4,5) | 5 | Sampled |
| Pericyte | 1 | All (1,2,3,4,5) | 5 | Sampled |
| OPC | 1 | All (1,2,3,4,5) | 5 | Sampled |
| Two neurons | 2 | 1 | 5 | Sampled |
| Two astrocytes | 2 | 1 | 5 | Sampled |
| Small cube | 10 | 3 | 5 | Sampled |
| Small cube | 10 | 1 | 20 | Sampled |
| Medium cube | 30 | 1 | 20 | Sampled |
| Large cube | 100 | 1 | 20 | Sampled |

In [ ]:
# Configuration parameters for each cell type
cell_configs = {
    "neuron":       {"mips": [0, 1, 2, 4], "samples": 3, "cell_max_size": 10000, "ncells": 1},
    "astrocyte":    {"mips": [0, 1, 2, 4], "samples": 3, "cell_max_size": 10000, "ncells": 1},
    "microglia":    {"mips": [0, 1, 2, 4], "samples": 3, "cell_max_size": 10000, "ncells": 1},
    "oligo":        {"mips": [0, 1, 2, 4], "samples": 3, "cell_max_size": 10000, "ncells": 1},
    "pericyte":     {"mips": [0, 1, 2, 4], "samples": 3, "cell_max_size": 10000, "ncells": 1},
    "OPC":          {"mips": [0, 1, 2, 4], "samples": 3, "cell_max_size": 10000, "ncells": 1},
}

In [ ]:
import yaml
import os
import random

output_dir = "../emimesh/config_files/sscp_configs"
os.makedirs(output_dir, exist_ok=True)

# Template configuration for single cell
def generate_config(cell_type, mip, cell_idx, cell_max_size, ncells=1):
    config = {
        "name": f"{cell_type}_mip{mip}_smoothed",
        "raw": {
            "cloudpath": "precomputed://gs://iarpa_microns/minnie/minnie65/seg_m1300",
            "position": "0-0-0", # Not used in this context
            "mip": mip,
            "size": 10000,
            "cell_type": cell_type,
            "cell_idx": cell_idx,
            "cell_padding": 100,
            "cell_table_name": "aibs_metamodel_celltypes_v661",
            "cell_max_size": cell_max_size
        },
        "meshing": {
            "envelopsize": 8
        }
    }

    # Follow readme simple processing setup
    if ncells == 1:
        config["processing"] = {
            "dx": 100,
            "operation": [
                "smooth iterations=1 radius=2",
                "erode radius=1"
            ]
        }
    else:
        config["raw"]["cell_keep_surrounding"] = True
        config["processing"] = {
            "dx": 100,
            "operation": [
                f"ncells ncells={ncells}",
                # "smooth iterations=1 radius=2",
                # "erode radius=1",
                "removeislands minsize=5000",    # Remove small noise
                "dilate radius=1",               # Close small gaps
                "smooth iterations=1 radius=1",  # Smooth boundaries
                "erode radius=1"                 # Create gaps between cells
            ]
        }
    
    return config

# Generate and save files
for cell_type, params in cell_configs.items():
    for cell_idx in range(params["samples"]):
        for mip in params["mips"]:
            # Include sample index in filename to avoid overwriting
            ncells = params.get("ncells", 1)
            if ncells > 1:
                filename = f"{cell_type}_sample{cell_idx}_mip{mip}_ncells{ncells}.yml"
            else:
                filename = f"{cell_type}_sample{cell_idx}_mip{mip}.yml"
            filepath = os.path.join(output_dir, filename)
            config_data = generate_config(cell_type, mip, cell_idx, params["cell_max_size"], ncells)

            with open(filepath, 'w') as f:
                yaml.dump(config_data, f, default_flow_style=False)
            
        print(f"Generated configs for {cell_type} sample {cell_idx}")

print("\nAll configuration files have been generated.")